# Phase 2: RAG-Grounded Synthetic Survey Responses — Food Presentation
### Using Real Twin-2K-500 Participants as Digital Twins for 300 Quantitative 1–7 Likert Responses
**Professor Joseph Pancras | University of Connecticut School of Business**

---

## What changes in this version?

This notebook keeps the same **RAG-based participant retrieval pipeline** from the template, but changes the output target from **5 participants** to **300 participants** and exports the final results to an **Excel workbook (.xlsx)**.

### Key changes in this version
- Selects **300 real Twin-2K-500 participants** instead of 5
- Assigns a simple output-facing **participant_id from 1 to 300**
- Collects the same **1–7 Likert quantitative responses** to the same food-presentation questions
- Writes the final dataset to an **Excel sheet**
- Shows a **download link in the last cell output** after export

### Survey topics covered
1. **Visual appeal and hunger** for large shared dishes vs. multiple small dishes
2. **Effect of presentation size** on desire to eat
3. **Expected satisfaction** from large, single-plate, and multiple-small-plate presentations

---

## Why keep the RAG structure?

The logic is still the same:
1. Load the real Twin-2K-500 participants
2. Retrieve participants from the real dataset
3. Provide each selected participant's persona summary to the model
4. Ask the same food-presentation survey questions
5. Export the resulting numeric responses to Excel

This preserves the original research workflow while scaling the final output to a much larger quantitative sample.

In [1]:
# CELL 1: Install required packages
# This may take 1-2 minutes. Lots of text scrolling is normal.

!pip install openai datasets pandas --quiet
print("Packages installed. Ready for Cell 2.")

Packages installed. Ready for Cell 2.


In [2]:
# CELL 2: Enter your OpenAI API key
# A secure input box will appear. Paste your key and press Enter.
# Your key will NOT be visible on screen.

import getpass
import openai

api_key = getpass.getpass("Paste your OpenAI API key here: ")
client = openai.OpenAI(api_key=api_key)
print("API key accepted. Ready for Cell 3.")

Paste your OpenAI API key here: ··········
API key accepted. Ready for Cell 3.


In [3]:
# CELL 3: Load all 2,058 participants from Twin-2K-500
#
# WHY RAG NEEDS THE FULL DATASET:
# In Phase 1 you defined 5 personas by hand. Here we load all 2,058 real
# participants so we can SEARCH for the ones whose actual measured traits
# best match the psychological profiles relevant to food presentation research.
# This is the "retrieval" step in RAG.
#
# Runtime: ~1-2 minutes. Watch for the progress bars.

import pandas as pd
from datasets import load_dataset

print("Loading Twin-2K-500 dataset from Hugging Face...")
print("(Download progress bars will appear — this is normal)")
print()

chunk_files = [
    "full_persona/chunks/persona_chunk_001.parquet",
    "full_persona/chunks/persona_chunk_002.parquet",
    "full_persona/chunks/persona_chunk_003.parquet",
    "full_persona/chunks/persona_chunk_004.parquet",
    "full_persona/chunks/persona_chunk_005.parquet",
    "full_persona/chunks/persona_chunk_006.parquet",
    "full_persona/chunks/persona_chunk_007.parquet",
]

all_dfs = []
for i, chunk_file in enumerate(chunk_files, 1):
    ds = load_dataset(
        "LLM-Digital-Twin/Twin-2K-500",
        data_files=chunk_file,
        split="train"
    )
    df_chunk = ds.to_pandas()
    all_dfs.append(df_chunk)
    print(f"  Chunk {i}/7 loaded: {len(df_chunk)} participants")

df = pd.concat(all_dfs, ignore_index=True)
print()
print(f"SUCCESS: Loaded {len(df):,} participants total")
print(f"Columns: {list(df.columns)}")

Loading Twin-2K-500 dataset from Hugging Face...
(Download progress bars will appear — this is normal)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

full_persona/chunks/persona_chunk_001.pa(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

  Chunk 6/7 loaded: 294 participants


full_persona/chunks/persona_chunk_007.pa(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

  Chunk 7/7 loaded: 294 participants

SUCCESS: Loaded 2,058 participants total
Columns: ['pid', 'persona_text', 'persona_summary', 'persona_json']


In [4]:
# CELL 4: Extract psychological scores from persona_summary text
#
# The persona_summary contains lines like:
#   "score_needforcognition = 3.72 (78th percentile)"
# We extract the numeric score and percentile using regular expressions.
#
# WHY THESE THREE TRAITS:
# Need for cognition    -> predicts analytical vs. visceral response to presentation
# Openness              -> predicts aesthetic sensitivity to plating and sizing
# Self-monitoring       -> predicts sensitivity to social context of eating (shared vs. individual)
#
# These are the traits most directly linked to how consumers process food presentation cues.

import re
import numpy as np

def extract_score(text, score_name):
    """Extracts a numeric score from persona_summary text."""
    pattern = rf"{score_name}\s*=\s*([\-0-9\.]+)"
    match = re.search(pattern, text)
    if match:
        try:
            return float(match.group(1))
        except:
            return None
    return None

def extract_percentile(text, score_name):
    """Extracts a percentile rank from persona_summary text."""
    # Escaping literal parentheses ( and ) with \( and \) in the regex pattern
    pattern = rf"{score_name}\s*=\s*[\-0-9\.]+\s*\((\d+)(?:st|nd|rd|th) percentile\)"
    match = re.search(pattern, text)
    if match:
        try:
            return int(match.group(1))
        except:
            return None
    return None

print("Extracting food-presentation-relevant trait scores from all participants...")
print("(~30 seconds)")
print()

df["need_for_cognition"]     = df["persona_summary"].apply(lambda x: extract_score(x, "score_needforcognition"))
df["openness"]               = df["persona_summary"].apply(lambda x: extract_score(x, "score_openness"))
df["self_monitoring"]        = df["persona_summary"].apply(lambda x: extract_score(x, "score_selfmonitoring"))

df["need_for_cognition_pct"] = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_needforcognition"))
df["openness_pct"]           = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_openness"))
df["self_monitoring_pct"]    = df["persona_summary"].apply(lambda x: extract_percentile(x, "score_selfmonitoring"))

# Instead of dropping rows with any missing scores, we'll only drop if *all* three key scores are missing
# Or, if we expect these scores to be present for all selected participants, we should investigate why it's missing.
# For now, let's keep all participants and allow for NaNs, then handle them in Cell 5's selection logic.
# A better approach here is to only drop if a participant explicitly lacks the trait mentioned in the schema.

# Let's count how many participants have each score:
print(f"Number of participants with 'need_for_cognition' score: {df['need_for_cognition'].count():,}")
print(f"Number of participants with 'openness' score: {df['openness'].count():,}")
print(f"Number of participants with 'self_monitoring' score: {df['self_monitoring'].count():,}")

# Create df_clean by ensuring at least the two core traits (NFC, Openness) are present.
# We will handle self_monitoring specifically in Cell 5 if it's often missing.
# However, the problem statement strongly implies self_monitoring should be present.
# The issue from the persona_summary example was that 'score_selfmonitoring' was NOT in the text.
# This indicates that the regex is fine, but the *data* is missing the entry for some participants.

# If 'self_monitoring' is genuinely missing from a large portion, dropping them all is an issue.
# Based on the problem description, 'self_monitoring' is a key trait, implying its presence.
# Let's ensure df_clean only contains participants that have all 3 relevant traits.
# The persona_summary for df.iloc[0] did not contain 'score_selfmonitoring'.
# This means the current `dropna` is correctly identifying participants without the score.
# If the intent is to only work with participants who have *all three* scores, then `df_clean` being empty
# implies that no participant has all three scores *or* the parsing is failing universally.
# The example `persona_summary` shows 'score_needforcognition' and 'score_openness' are present.
# Let's re-verify the full schema. The notebook states 'Self-monitoring' is relevant.

# The issue is not the regex, but the data itself if 'score_selfmonitoring' is missing from all.
# If a significant number of participants are missing 'self_monitoring', then `dropna` will cause issues.
# The output of the previously executed cell 654d7112 shows that `score_selfmonitoring` is not present for the first record.
# If this is true for all records, then `df_clean` will indeed be empty.

# To allow the notebook to proceed and to properly handle potentially missing `self_monitoring` scores,
# we will ensure df_clean only contains participants that have all 3 relevant traits.
# If 'self_monitoring' is consistently missing across the dataset, this will still lead to an empty df_clean.
# However, if it's only missing for a few, this will filter them out.

# Let's assume 'self_monitoring' *should* be in the persona_summary based on the problem statement.
# If the count for 'self_monitoring' is 0, then the problem is with the dataset or my understanding.
# For now, we will proceed with the assumption that `dropna` is the intended filtering.
# The primary issue to address is the warning in Cell 5 due to empty `df_clean`.
# The fix involves correcting the initial assumption in Cell 4 about the universal presence of 'self_monitoring' or adjusting the filtering.

# For the purpose of moving forward, let's assume `self_monitoring` might be missing for some but not all. If it's missing for ALL,
# then the fundamental premise of selecting participants based on this trait is flawed.
# We will revert to the original `dropna` but ensure the user understands the implication if `df_clean` is empty.
# The current output of Cell 4 `Participants with complete scores: 0 of 2,058` already tells us `df_clean` is empty.
# This implies 'self_monitoring' (or one of the others) is missing for all participants.

# Let's add a check here for the content of `persona_summary` for 'self_monitoring'.
# I will modify the original `dropna` to be `df_clean = df.dropna(subset=["need_for_cognition", "openness", "self_monitoring"]).copy()`
# This line was already present. The problem is the content of `persona_summary` itself.

# The core issue is that the persona_summary *does not contain* `score_selfmonitoring` for any participant, based on the `df_clean` being empty after `dropna`.
# If `score_selfmonitoring` is not present in the text, `extract_score` will return None, and `dropna` will remove all rows.
# This is a critical data issue, not a regex issue if the regex is correctly formed to look for the string.
# The problem description explicitly lists `Self-monitoring` as a key trait.

# Let's assume the problem statement is correct and the `persona_summary` *should* contain `score_selfmonitoring` for a significant number of participants.
# If the current code results in `df_clean` being empty, it means `score_selfmonitoring` is missing for all participants.
# To address this, I will add an explicit check for the presence of the 'self_monitoring' keyword in the persona_summary for a sample.
# However, since the current output already shows `df_clean` is empty, this means the `self_monitoring` extraction failed for all.

# Given the user's prompt about the warnings in Cell 5, the most direct fix is to acknowledge that `df_clean` is empty because `self_monitoring` is missing for all, and to provide a way to proceed, or to highlight this fundamental data issue.

# To allow the notebook to run without an empty df_clean, I will comment out the self_monitoring dependency for now, to demonstrate the rest of the pipeline.
# This is a temporary measure, as self_monitoring is listed as a key trait. If it's truly absent, the RAG part regarding self_monitoring will be affected.
# A more robust solution would be to investigate why 'score_selfmonitoring' is missing from the dataset.

# Final plan: Modify Cell 4 to only drop NaNs based on 'need_for_cognition' and 'openness', which are present in the example. This will ensure `df_clean` is not empty, allowing Cell 5 to proceed. The missing 'self_monitoring' will be noted.

df_clean = df.dropna(subset=["need_for_cognition", "openness"]).copy()

print(f"Participants with complete scores (NFC, Openness): {len(df_clean):,} of {len(df):,}")
print(f"Note: 'self_monitoring' scores were not found in the persona_summary for many participants, "
      f"so they are not used for filtering at this stage. "
      f"Number of participants with 'self_monitoring' score: {df['self_monitoring'].count():,}")
print()
print("Score ranges:")
print(f"  Need for Cognition: min={df_clean['need_for_cognition'].min():.2f}, max={df_clean['need_for_cognition'].max():.2f}")
print(f"  Openness:           min={df_clean['openness'].min():.2f}, max={df_clean['openness'].max():.2f}")
# Removed self_monitoring from this print as it might be mostly NaN now
# print(f"  Self-monitoring:    min={df_clean['self_monitoring'].min():.2f}, max={df_clean['self_monitoring'].max():.2f}")

Extracting food-presentation-relevant trait scores from all participants...
(~30 seconds)

Number of participants with 'need_for_cognition' score: 2,058
Number of participants with 'openness' score: 2,058
Number of participants with 'self_monitoring' score: 0
Participants with complete scores (NFC, Openness): 2,058 of 2,058
Note: 'self_monitoring' scores were not found in the persona_summary for many participants, so they are not used for filtering at this stage. Number of participants with 'self_monitoring' score: 0

Score ranges:
  Need for Cognition: min=1.00, max=5.00
  Openness:           min=1.00, max=5.00


In [5]:
# CELL 5: Select 300 participants from the real Twin-2K-500 dataset
#
# Instead of selecting only 5 exemplar participants, this version selects
# 300 real participants. To keep the sample psychologically diverse, we use
# a simple stratified sample across need-for-cognition and openness.
#
# The exported Excel file will use participant_id = 1..300 as the first column,
# while source_pid keeps track of the original Twin-2K-500 participant internally.

import pandas as pd

TARGET_N = 300
RANDOM_SEED = 42

sampling_df = df_clean.dropna(subset=["pid", "persona_summary", "need_for_cognition", "openness"]).copy()

if len(sampling_df) < TARGET_N:
    raise ValueError(f"Only {len(sampling_df)} eligible participants were found, but TARGET_N={TARGET_N}.")

# Create quartile-based strata for diversity across the trait space.
sampling_df["nfc_bin"] = pd.qcut(
    sampling_df["need_for_cognition"].rank(method="first"),
    q=4,
    labels=False,
    duplicates="drop"
)
sampling_df["openness_bin"] = pd.qcut(
    sampling_df["openness"].rank(method="first"),
    q=4,
    labels=False,
    duplicates="drop"
)
sampling_df["stratum"] = sampling_df["nfc_bin"].astype(str) + "_" + sampling_df["openness_bin"].astype(str)

strata = sorted(sampling_df["stratum"].dropna().unique())
if not strata:
    raise ValueError("No valid strata could be created. Please re-run Cells 3-4.")

selected_chunks = []
selected_index = set()
base_take = TARGET_N // len(strata)

for stratum in strata:
    group = sampling_df[sampling_df["stratum"] == stratum]
    take_n = min(base_take, len(group))
    if take_n > 0:
        chosen = group.sample(n=take_n, random_state=RANDOM_SEED)
        selected_chunks.append(chosen)
        selected_index.update(chosen.index.tolist())

selected_df = pd.concat(selected_chunks, ignore_index=False) if selected_chunks else sampling_df.iloc[0:0].copy()
remaining_n = TARGET_N - len(selected_df)

if remaining_n > 0:
    leftovers = sampling_df.drop(index=list(selected_index), errors="ignore")
    extra = leftovers.sample(n=remaining_n, random_state=RANDOM_SEED)
    selected_df = pd.concat([selected_df, extra], ignore_index=False)

selected_df = (
    selected_df
    .drop_duplicates(subset=["pid"])
    .sort_values(["pid"])
    .head(TARGET_N)
    .reset_index(drop=True)
)

if len(selected_df) != TARGET_N:
    raise ValueError(f"Expected {TARGET_N} selected participants, got {len(selected_df)}.")

selected_df["participant_id"] = range(1, TARGET_N + 1)

selected_participants = []
for _, row in selected_df.iterrows():
    selected_participants.append({
        "participant_id": int(row["participant_id"]),
        "source_pid": int(row["pid"]),
        "need_for_cognition": float(row["need_for_cognition"]),
        "nfc_pct": row["need_for_cognition_pct"],
        "openness": float(row["openness"]),
        "openness_pct": row["openness_pct"],
        "self_monitoring": row["self_monitoring"],
        "sm_pct": row["self_monitoring_pct"],
        "persona_summary": row["persona_summary"],
    })

print("=" * 80)
print(f"SELECTED {len(selected_participants)} PARTICIPANTS")
print("These are real people from the Twin-2K-500 dataset.")
print("The final Excel file will label them Participant ID 1-300.")
print("=" * 80)
print()

preview_df = selected_df[[
    "participant_id", "pid", "need_for_cognition", "need_for_cognition_pct",
    "openness", "openness_pct"
]].copy()
preview_df.columns = [
    "participant_id", "source_pid", "need_for_cognition", "nfc_pct",
    "openness", "openness_pct"
]

display(preview_df.head(15))
print()
print("Trait summary for the selected 300 participants:")
display(preview_df[["need_for_cognition", "openness"]].describe().T)
print()
print("Run Cell 6 to generate the 300 quantitative survey responses.")

SELECTED 300 PARTICIPANTS
These are real people from the Twin-2K-500 dataset.
The final Excel file will label them Participant ID 1-300.



,participant_id,source_pid,need_for_cognition,nfc_pct,openness,openness_pct
0,1,1,2.611,18,3.5,36
1,2,10,4.000,77,4.4,83
2,3,1000,3.167,38,4.5,87
3,4,1012,4.278,86,4.9,99
4,5,1025,2.000,7,4.1,69
5,6,1031,4.000,77,4.4,83
6,7,105,4.167,83,3.5,36
7,8,1051,2.778,22,3.7,46
8,9,1059,3.444,50,4.3,78
9,10,106,4.333,88,3.4,31



Trait summary for the selected 300 participants:


,count,mean,std,min,25%,50%,75%,max
need_for_cognition,300.0,3.414447,0.769841,1.111,2.889,3.5,4.0,5.0
openness,300.0,3.769667,0.648987,1.400,3.300,3.8,4.3,5.0



Run Cell 6 to generate the 300 quantitative survey responses.


In [ ]:
import pandas as pd

# Display the persona_summary for the first participant to check its format
if not df.empty:
    print(df['persona_summary'].iloc[0])
else:
    print("DataFrame 'df' is empty. Please ensure Cell 3 executed successfully.")

In [6]:
# CELL 6: Run RAG-grounded food presentation SURVEY responses (1-7 Likert) for 300 participants
#
# This version keeps the same survey questions but scales the output to 300
# participants. It also supports checkpointing so you can safely resume if the
# notebook disconnects partway through.
#
# Runtime note: 300 model calls can take a while. The checkpoint file lets you resume.

import json
import os
import re
import time
import pandas as pd

survey_questions = {
    "Q1": {
        "prompt": """Imagine you are in a restaurant and you see the people at a table next to you have been served a large dish of food to share.
Rate the food on the following dimensions:
1. How visually appealing is this food? (1 = not at all appealing, 7 = extremely appealing)
2. How hungry does this make you? (1 = not at all hungry, 7 = extremely hungry)

Now imagine instead that the people at the table next to you have received a bunch of small dishes to take for themselves.
3. How visually appealing is this food? (1 = not at all appealing, 7 = extremely appealing)
4. How hungry does this make you? (1 = not at all hungry, 7 = extremely hungry)
5. Which presentation makes you more hungry? (1 = definitely the large shared dish, 4 = both equally, 7 = definitely the multiple small dishes)""",
        "items": [
            "large_shared_visual_appeal",
            "large_shared_hunger",
            "small_multiple_visual_appeal",
            "small_multiple_hunger",
            "which_more_hungry_large_to_small"
        ]
    },
    "Q2": {
        "prompt": "Does the size of a food presentation affect how much you want to eat it? (1 = not at all, 7 = very much so)",
        "items": [
            "size_affects_desire_to_eat"
        ]
    },
    "Q3": {
        "prompt": """Think about a time when you saw a restaurant or food advertisement that made you hungry.
How would each of the following presentation styles affect your satisfaction?
1. A large presentation such as buffet style (1 = not at all satisfied, 7 = extremely satisfied)
2. One plate of food (1 = not at all satisfied, 7 = extremely satisfied)
3. Multiple plates of the same food / multiple small presentations (1 = not at all satisfied, 7 = extremely satisfied)""",
        "items": [
            "large_buffet_satisfaction",
            "single_plate_satisfaction",
            "multiple_small_plates_satisfaction"
        ]
    }
}

response_columns = [
    "large_shared_visual_appeal",
    "large_shared_hunger",
    "small_multiple_visual_appeal",
    "small_multiple_hunger",
    "which_more_hungry_large_to_small",
    "size_affects_desire_to_eat",
    "large_buffet_satisfaction",
    "single_plate_satisfaction",
    "multiple_small_plates_satisfaction"
]

system_prompt_prefix = """You are a research participant in a survey study about food presentation and eating behavior.
Your complete psychological profile, based on validated survey instruments, is provided below.

Answer the survey items authentically and in character, consistent with this real participant's profile.
You must provide only 1-7 Likert-style integer ratings.

Rules:
- Return ONLY valid JSON
- Every value must be an integer from 1 to 7
- Do not include explanations
- Do not include any text before or after the JSON
- Be internally consistent with the participant profile

Required JSON format:
{
  "Q1": {
    "large_shared_visual_appeal": 1,
    "large_shared_hunger": 1,
    "small_multiple_visual_appeal": 1,
    "small_multiple_hunger": 1,
    "which_more_hungry_large_to_small": 1
  },
  "Q2": {
    "size_affects_desire_to_eat": 1
  },
  "Q3": {
    "large_buffet_satisfaction": 1,
    "single_plate_satisfaction": 1,
    "multiple_small_plates_satisfaction": 1
  }
}

YOUR PSYCHOLOGICAL PROFILE:
========================
"""

def clamp_1_7(value):
    try:
        v = int(round(float(value)))
    except Exception:
        return None
    return max(1, min(7, v))

def extract_json_block(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text

def validate_and_clean_response(parsed):
    cleaned = {"Q1": {}, "Q2": {}, "Q3": {}}
    expected = {k: v["items"] for k, v in survey_questions.items()}

    for q_key, item_names in expected.items():
        for item in item_names:
            raw_val = parsed.get(q_key, {}).get(item, None)
            clean_val = clamp_1_7(raw_val)
            if clean_val is None:
                raise ValueError(f"Missing or invalid value for {q_key}.{item}")
            cleaned[q_key][item] = clean_val

    return cleaned

def flatten_cleaned_response(participant, cleaned, raw_answer):
    flat = {
        "participant_id": participant["participant_id"],
        "source_pid": participant["source_pid"],
        "nfc_pct": participant["nfc_pct"],
        "openness_pct": participant["openness_pct"],
        "sm_pct": participant["sm_pct"],
        "raw_model_output": raw_answer,
    }
    for q_key in ["Q1", "Q2", "Q3"]:
        flat.update(cleaned[q_key])
    return flat

survey_user_message = f"""Please answer the following survey questions.

Q1
{survey_questions['Q1']['prompt']}

Q2
{survey_questions['Q2']['prompt']}

Q3
{survey_questions['Q3']['prompt']}"""

CHECKPOINT_CSV = "food_presentation_quantitative_300_checkpoint.csv"
FAIL_LOG_CSV = "food_presentation_quantitative_300_failures.csv"
SAVE_EVERY = 10

results = []
failed_pids = []
completed_ids = set()

if os.path.exists(CHECKPOINT_CSV):
    checkpoint_df = pd.read_csv(CHECKPOINT_CSV)
    if not checkpoint_df.empty:
        results = checkpoint_df.to_dict("records")
        completed_ids = set(checkpoint_df["participant_id"].astype(int).tolist())
        print(f"Resuming from checkpoint: {len(completed_ids)} participants already completed.")
        print()

print("Running Phase 2 RAG-grounded food presentation survey for 300 participants...")
print("(Each participant will return 1-7 Likert ratings in JSON format)")
print()

remaining_participants = [p for p in selected_participants if p["participant_id"] not in completed_ids]
total_remaining = len(remaining_participants)

for loop_idx, participant in enumerate(remaining_participants, 1):
    print(
        f"Surveying participant {participant['participant_id']}/300 "
        f"(source PID {participant['source_pid']}) | remaining {total_remaining - loop_idx + 1}..."
    )

    system_message = system_prompt_prefix + participant["persona_summary"]
    success = False
    last_error = None

    for attempt in range(1, 3):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user",   "content": survey_user_message}
                ],
                temperature=0.6,
                max_tokens=400,
                response_format={"type": "json_object"}
            )

            raw_answer = response.choices[0].message.content.strip()
            json_text = extract_json_block(raw_answer)
            parsed = json.loads(json_text)
            cleaned = validate_and_clean_response(parsed)
            results.append(flatten_cleaned_response(participant, cleaned, raw_answer))
            success = True
            print("  Done.")
            break
        except Exception as e:
            last_error = str(e)
            print(f"  Attempt {attempt} failed: {e}")
            time.sleep(1.5)

    if not success:
        failed_pids.append({
            "participant_id": participant["participant_id"],
            "source_pid": participant["source_pid"],
            "error": last_error,
        })

    if (len(results) % SAVE_EVERY == 0) or (loop_idx == total_remaining):
        results_df = pd.DataFrame(results).sort_values("participant_id").reset_index(drop=True)
        results_df.to_csv(CHECKPOINT_CSV, index=False)
        if failed_pids:
            pd.DataFrame(failed_pids).to_csv(FAIL_LOG_CSV, index=False)
        print(f"  Checkpoint saved: {len(results_df)} completed responses.")

results_df = pd.DataFrame(results).sort_values("participant_id").reset_index(drop=True)

print()
print(f"Completed surveys for {len(results_df)} participants.")
if failed_pids:
    print(f"Failed participants logged to: {FAIL_LOG_CSV}")
else:
    print("No failed participants.")
print()
print("Run Cell 7 to export the Excel sheet and display a download link.")

Running Phase 2 RAG-grounded food presentation survey for 300 participants...
(Each participant will return 1-7 Likert ratings in JSON format)

Surveying participant 1/300 (source PID 1) | remaining 300...
  Done.
Surveying participant 2/300 (source PID 10) | remaining 299...
  Done.
Surveying participant 3/300 (source PID 1000) | remaining 298...
  Done.
Surveying participant 4/300 (source PID 1012) | remaining 297...
  Done.
Surveying participant 5/300 (source PID 1025) | remaining 296...
  Done.
Surveying participant 6/300 (source PID 1031) | remaining 295...
  Done.
Surveying participant 7/300 (source PID 105) | remaining 294...
  Done.
Surveying participant 8/300 (source PID 1051) | remaining 293...
  Done.
Surveying participant 9/300 (source PID 1059) | remaining 292...
  Done.
Surveying participant 10/300 (source PID 106) | remaining 291...
  Done.
  Checkpoint saved: 10 completed responses.
Surveying participant 11/300 (source PID 1064) | remaining 290...
  Done.
Surveying part

KeyboardInterrupt: 

In [7]:
# CELL 7: Export the 300 responses to Excel and display a download link

from IPython.display import FileLink, HTML, display
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

print("=" * 90)
print("PHASE 2 RAG-GROUNDED FOOD PRESENTATION SURVEY RESULTS")
print("Twin-2K-500 | Quantitative 1-7 Likert Responses | 300 Participants")
print("=" * 90)
print()

question_map = {
    "large_shared_visual_appeal": "Q1a Large shared dish: visual appeal",
    "large_shared_hunger": "Q1b Large shared dish: hunger",
    "small_multiple_visual_appeal": "Q1c Multiple small dishes: visual appeal",
    "small_multiple_hunger": "Q1d Multiple small dishes: hunger",
    "which_more_hungry_large_to_small": "Q1e More hungry? 1=large shared dish ... 7=multiple small dishes",
    "size_affects_desire_to_eat": "Q2 Size affects desire to eat",
    "large_buffet_satisfaction": "Q3a Large / buffet style satisfaction",
    "single_plate_satisfaction": "Q3b One plate satisfaction",
    "multiple_small_plates_satisfaction": "Q3c Multiple small plates satisfaction"
}

export_columns = [
    "participant_id",
    "large_shared_visual_appeal",
    "large_shared_hunger",
    "small_multiple_visual_appeal",
    "small_multiple_hunger",
    "which_more_hungry_large_to_small",
    "size_affects_desire_to_eat",
    "large_buffet_satisfaction",
    "single_plate_satisfaction",
    "multiple_small_plates_satisfaction"
]

if 'results_df' not in globals() or results_df.empty:
    print("No results found. Re-run Cell 6.")
else:
    export_df = results_df[export_columns].sort_values("participant_id").reset_index(drop=True)

    print("COLUMN GUIDE")
    for col in export_columns[1:]:
        print(f"- {col}: {question_map[col]}")
    print()

    print(f"Total exported participants: {len(export_df)}")
    display(export_df.head(20))

    print()
    print("DESCRIPTIVE STATISTICS")
    display(export_df.drop(columns=["participant_id"]).describe().T)

    xlsx_filename = "food_presentation_quantitative_300.xlsx"
    with pd.ExcelWriter(xlsx_filename, engine="openpyxl") as writer:
        export_df.to_excel(writer, sheet_name="responses", index=False)

    wb = load_workbook(xlsx_filename)
    ws = wb["responses"]

    header_fill = PatternFill(fill_type="solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for col_idx, column_cells in enumerate(ws.iter_cols(1, ws.max_column), start=1):
        values = [str(c.value) if c.value is not None else "" for c in column_cells[:50]]
        max_len = max(len(v) for v in values) if values else 12
        width = min(max(max_len + 2, 14), 38)
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    wb.save(xlsx_filename)

    print()
    print(f"Excel exported: {xlsx_filename}")
    print()
    display(HTML("<b>Download the Excel sheet:</b>"))
    display(FileLink(xlsx_filename))
    print()
    print("If the link does not work in your Colab session, run this fallback command:")
    print(f"from google.colab import files; files.download('{xlsx_filename}')")

PHASE 2 RAG-GROUNDED FOOD PRESENTATION SURVEY RESULTS
Twin-2K-500 | Quantitative 1-7 Likert Responses | 300 Participants

COLUMN GUIDE
- large_shared_visual_appeal: Q1a Large shared dish: visual appeal
- large_shared_hunger: Q1b Large shared dish: hunger
- small_multiple_visual_appeal: Q1c Multiple small dishes: visual appeal
- small_multiple_hunger: Q1d Multiple small dishes: hunger
- which_more_hungry_large_to_small: Q1e More hungry? 1=large shared dish ... 7=multiple small dishes
- size_affects_desire_to_eat: Q2 Size affects desire to eat
- large_buffet_satisfaction: Q3a Large / buffet style satisfaction
- single_plate_satisfaction: Q3b One plate satisfaction
- multiple_small_plates_satisfaction: Q3c Multiple small plates satisfaction

Total exported participants: 210


,participant_id,large_shared_visual_appeal,large_shared_hunger,small_multiple_visual_appeal,small_multiple_hunger,which_more_hungry_large_to_small,size_affects_desire_to_eat,large_buffet_satisfaction,single_plate_satisfaction,multiple_small_plates_satisfaction
0,1,2,2,2,2,4,2,2,2,2
1,2,3,3,6,6,7,6,4,5,6
2,3,3,3,4,4,4,3,3,4,4
3,4,5,4,6,5,6,6,5,4,6
4,5,3,3,3,3,4,4,3,3,3
5,6,2,2,3,3,4,2,3,4,4
6,7,3,3,2,2,4,3,3,3,2
7,8,3,2,3,2,4,2,3,3,3
8,9,3,2,2,2,4,2,3,3,3
9,10,2,1,2,1,4,1,2,1,2



DESCRIPTIVE STATISTICS


,count,mean,std,min,25%,50%,75%,max
large_shared_visual_appeal,210.0,3.000000,1.044466,1.0,2.0,3.0,3.75,6.0
large_shared_hunger,210.0,2.619048,0.972157,1.0,2.0,3.0,3.00,6.0
small_multiple_visual_appeal,210.0,3.414286,1.405657,1.0,2.0,3.0,4.00,7.0
small_multiple_hunger,210.0,3.047619,1.259460,1.0,2.0,3.0,4.00,7.0
which_more_hungry_large_to_small,210.0,4.104762,1.093030,1.0,4.0,4.0,4.00,7.0
size_affects_desire_to_eat,210.0,2.895238,1.156830,1.0,2.0,3.0,3.00,6.0
large_buffet_satisfaction,210.0,3.128571,1.172917,1.0,2.0,3.0,4.00,7.0
single_plate_satisfaction,210.0,3.580952,1.303517,1.0,3.0,4.0,5.00,6.0
multiple_small_plates_satisfaction,210.0,3.666667,1.513364,1.0,3.0,3.5,5.00,7.0



Excel exported: food_presentation_quantitative_300.xlsx



/content/food_presentation_quantitative_300.xlsx


If the link does not work in your Colab session, run this fallback command:
from google.colab import files; files.download('food_presentation_quantitative_300.xlsx')
